# Milestone 2 — Khung chạy parser (chưa triển khai)

Notebook này là template cho **Danh ghép cuối**, khác milestone2_prepare.ipynb đã khôi phục raw. Các APIs parser hiện báo NotImplementedError. Chưa có kết quả M2, benchmark hay tỷ lệ thành công.

Đọc src/parsing/contracts.py và docstring từng prototype để xem schema/cách làm. Chỉ bật RUN_PARSER sau khi nhận các modules và chạy integration thật. Colab dùng CPU / accelerator None; checkout phải có source mới, notebook không tự pull đè code.

Prototype/docstring thuật toán và template là hướng tham khảo; mục tiêu là hoàn thiện toàn bộ task và tiêu chí nghiệm thu. Nếu có hướng tốt hơn, AI so sánh lợi ích/đánh đổi, dependencies, ảnh hưởng tích hợp và cách kiểm chứng rồi hỏi người phụ trách chọn hướng trước khi làm phần phụ thuộc; tiếp tục phần độc lập trong lúc chờ, không hỏi lại chi tiết nhỏ hoặc lựa chọn đã chốt. Ưu tiên giữ nguyên schema, kiểu dữ liệu và API; nếu không cần đổi mà vẫn hoàn thành tốt task thì càng tốt. Không khuyến khích đổi, chỉ đổi khi thật sự cần thiết và hạn chế sửa phần chung để tránh conflict. Phối hợp Danh/các owner bị ảnh hưởng và cập nhật đồng bộ consumer, fixtures, tests, notebook, tài liệu. Không chỉ điền các hàm có sẵn hoặc giảm yêu cầu để hoàn thành template.


In [ ]:
from pathlib import Path

PROJECT_DIR = None
DATASET_NAME = 'Danh2'
PREPARATION_DIR = Path('/content/drive/MyDrive/arXivProject/preparationM2')
# Gói raw đã prepare nếu có, KHÔNG phải corpus.zip backup M1. Notebook prepare không xuất ZIP.
PREPARED_ARCHIVE = PREPARATION_DIR / 'transport' / 'Danh2-<snapshot>.zip'
LOCAL_WORK_DIR = Path('/content/arxiv-m2-work')
RAW_CORPUS_ROOT = LOCAL_WORK_DIR / 'raw' / DATASET_NAME
OUTPUT_ROOT = LOCAL_WORK_DIR / 'parsed' / DATASET_NAME
DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/arXivProject/Milestone2/parsed') / DATASET_NAME
STAGE_INPUT = False  # Bật để copy ZIP đã prepare vào thư mục local mới.
RUN_PARSER = False  # Các prototypes chưa triển khai.
RESUME = False
STRICT = True
CODE_REVISION = 'unrecorded'
REFERENCE_REVIEW_PATH = None  # Dùng bản local của file review nếu có.


## 1. Mở checkout và kiểm cấu hình

Nếu chạy Colab, tự clone/đưa đúng checkout đã ghép lên runtime rồi mount Drive bằng google.colab.drive.mount('/content/drive') trước khi chạy parser. Không cần mount Drive để đọc bộ khung. Output phải khác cây raw; không có module owner nào tự ghi sửa raw.

Giữ bước M1 → preparationM2 ở notebook prepare riêng. Chọn đúng PREPARED_ARCHIVE và bật STAGE_INPUT cho lần đầu; local root đã tồn tại thì tắt stage để dùng lại hoặc chọn đích mới. Output/checkpoint ở local. Batch backup Drive thuộc DA3, chưa có trong template; không coi local output là bản bền vững.


In [ ]:
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python >=3.11.')
if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'src/parsing/contracts.py').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('Đặt PROJECT_DIR tới checkout có bộ khung M2.')
sys.path.insert(0, str(Path(PROJECT_DIR) / 'src'))
from parsing.contracts import CONTRACT_VERSION, ParseConfig
from parsing.pipeline import run
print('Contract:', CONTRACT_VERSION)
print('Raw:', RAW_CORPUS_ROOT)
print('Output:', OUTPUT_ROOT)
print('RUN_PARSER:', RUN_PARSER, '— prototypes chưa có nghiệp vụ.')


## Stage raw đã prepare về local

Copy archive lớn một lần, kiểm SHA-256 và giải nén an toàn. Không chạy restore M1. Cell này không full-validate raw; input adapter kiểm local corpus một lần đầu mỗi run, kể cả khi dùng lại local. Không ghi sửa raw trong run. Cần đủ disk cho archive + raw + outputs/batch tạm.


In [ ]:
if STAGE_INPUT:
    import json
    from preparation.transport import stage_prepared_archive
    receipt_path = Path(str(PREPARED_ARCHIVE) + '.receipt.json')
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    if receipt.get('kind') != 'm2-prepared-transport-1.0':
        raise ValueError('Cần archive raw đã prepare, không phải backup M1.')
    RAW_CORPUS_ROOT = stage_prepared_archive(
        PREPARED_ARCHIVE, RAW_CORPUS_ROOT, expected_sha256=receipt['sha256'])
    completion = json.loads((RAW_CORPUS_ROOT / 'preparation.json').read_text(encoding='utf-8'))
    if completion.get('snapshot_id') != receipt['snapshot_id']:
        raise ValueError('Snapshot của archive không khớp receipt; không chạy parser.')
    print('Raw local:', RAW_CORPUS_ROOT, '— parser còn phải kiểm full validation.')
else:
    print('Chưa stage. Khi chạy parser, RAW_CORPUS_ROOT phải là raw local đã prepare.')
print('Backup Drive dự kiến:', DRIVE_BACKUP_ROOT)
print('Batch backup parser chưa triển khai; /content/ có thể mất khi runtime ngắt.')


## 2. Chạy sau khi tích hợp

Pipeline phải tự kiểm read-back raw, log snapshot, giữ mọi expected version, finalize status và kiểm output. Không bỏ blocked/failed khỏi mẫu số. Run thường có thể resume; benchmark chính thức dùng benchmark.benchmark_run trên output mới và Colab CPU, đo riêng với setup/prepare.


In [ ]:
if not RUN_PARSER:
    print('CHƯA CHẠY: đây là template; tiếp tục triển khai các module theo phân công.')
else:
    if CODE_REVISION == 'unrecorded':
        raise RuntimeError('Ghi code revision thật trước khi chạy.')
    config = ParseConfig(strict=STRICT, code_revision=CODE_REVISION,
                         reference_review_path=REFERENCE_REVIEW_PATH)
    result = run(RAW_CORPUS_ROOT, OUTPUT_ROOT, config, resume=RESUME)
    print(result)
    if result.status != 'success':
        print('Chưa đạt toàn phạm vi; kiểm manifest/diagnostics, không gọi đây là M2 hoàn thành.')


## 3. Nghiệm thu của Danh và cả nhóm

- Mở một publication nhiều file, nhiều versions: source -> cây -> chuẩn hóa -> shared elements -> edges -> unified bib/cites. So tay với source/PDF khi có.
- Kiểm all-no_tex/blocked/partial đúng trạng thái; đọc lại output/JSON key map/occurrences, kiểm resume invalidation và lỗi ghi bằng fixtures.
- Chạy S1–S13 thực sau merge; fixtures không thay coverage corpus thật. Nhận phản hồi bàn giao của bốn người theo mẫu mục 12 phân công, có kiểm tay/reviewer.
- Fresh benchmark CPU Colab + raw measurements, Book execute/build sinh hình/bảng và S14 trên public Pages. Hai link Moodle đúng hạn, không nộp data M2.
- Dọn nháp/duplicate có evidence theo prompt tích hợp mục 15 phân công. Không xóa test fail/raw hoặc coi placeholders là implementation.

Chưa có việc nào ở checklist này được xác nhận chỉ bởi notebook template.
